# 03 - Erro com pontos flutuantes
Vamos aprender sobre como usar os erros de ponto flutuante para resolução de problemas numéricos.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana3
```

Faça o checkout nessa nova branch:

```bash
git checkout semana3
```

<hr />

## Atividade 1
A função exponencial natural pode ser definida pelo limite:
$$
e^x=\lim_{n\to\infty}\left(1+\frac{x}{n}\right)^n,
$$
mas também é dada pela **série de Maclaurin**:
$$
e^x=\sum_{n=0}^{\infty}\frac{x^n}{n!}
=1+\frac{x}{1!}+\frac{x^2}{2!}+\frac{x^3}{3!}+\cdots
$$

Implemente em **Python** o cálculo de $e^x$ pela série, interrompendo a soma quando o termo ficar menor que o limite prático de contribuição, usando a precisão de máquina como critério.

In [1]:
import math
import sys

def exp_series(x, atol=0.0):
    """ Aproxima e^x pela série de Maclaurin com critério de parada numérico. """
    eps = sys.float_info.epsilon
    s = 1.0
    term = 1.0
    n = 0
    tol_abs = max(atol, eps)

    while True:
        n += 1
        term *= x / n
        s += term
        if abs(term) < eps * abs(s) or abs(term) < tol_abs:
            break
        if n > 10_000:
            break
    return s, n, term

# Demonstração
for val in [1.0, 5.0, -2.0]:
    approx, nterms, last = exp_series(val)
    print(f"x={val:+g} -> e^x ≈ {approx:.16g} (math.exp={math.exp(val):.16g}, termos={nterms})")

x=+1 -> e^x ≈ 2.718281828459046 (math.exp=2.718281828459045, termos=18)
x=+5 -> e^x ≈ 148.4131591025766 (math.exp=148.4131591025766, termos=33)
x=-2 -> e^x ≈ 0.1353352832366127 (math.exp=0.1353352832366127, termos=24)


## Atividade 2

Implemente:
$$
e^x\approx\left(1+\frac{x}{n}\right)^n
$$
com $n$ crescente, e:
1. Explique por que, para $x<0$ e $n$ muito grande, pode ocorrer **cancelamento catastrófico**;
2. Proponha um critério de parada numérico para encerrar o crescimento de $n$ sem perder precisão.

In [2]:
import math


def exp_aprox(x: float, n: int) -> float:
    return (1 + x / n) ** n


def exp_aprox_robusto(x: float, tol: float = 1e-7, max_n: int = 1_000_000) -> float:
    
    if x < 0:
        return 1.0 / exp_aprox_robusto(-x, tol, max_n)

    n = 1
    val_anterior = exp_aprox(x, n)

    while n < max_n:
        n *= 2
        val_atual = exp_aprox(x, n)

        
        if abs(val_atual - val_anterior) / abs(val_atual) < tol:
            break

        val_anterior = val_atual

    return val_atual


def main():
    x = -10.0
    print(f"Valor exato (math.exp): {math.exp(x)}")
    print(f"Aproximação direta (n=1000): {exp_aprox(x, 1000)}")
    print(f"Aproximação robusta: {exp_aprox_robusto(x)}")


if __name__ == "__main__":
    main()

Valor exato (math.exp): 4.5399929762484854e-05
Aproximação direta (n=1000): 4.317124741065786e-05
Aproximação robusta: 4.540209463768425e-05


### 1. Ocorrência de Cancelamento Catastrófico ($x < 0$ e $n$ elevado)

Quando $x < 0$, a expressão assume a forma:

$$\left(1 - \frac{\vert{}x\vert{}}{n}\right)^n$$

Para valores de $n$ muito grandes:

* **Perda de precisão na base:** O termo $\frac{\vert{}x\vert{}}{n}$ torna-se extremamente pequeno em relação a $1$. A operação $1 - \frac{\vert{}x\vert{}}{n}$ realiza a subtração entre dois números de ordens de grandeza muito próximas, resultando na perda de algarismos significativos devido à precisão finita do sistema de ponto flutuante (*IEEE 754*).
* **Amplificação da imprecisão:** A imprecisão gerada na base é posteriormente amplificada ao elevar a expressão ao expoente $n$, degradando o resultado final.

---

### 2. Critérios de Parada Numéricos para Preservação de Precisão

Para encerrar o crescimento de $n$ com estabilidade numérica, adotam-se os seguintes critérios:

* **Critério de Erro Relativo (Estagnação):**
Acompanha-se a variação entre iterações sucessivas (por exemplo, dobrando $n$ a cada passo) e interrompe-se a execução quando a variação relativa for menor do que uma tolerância especificada $\epsilon$:
$$\frac{\vert{}y_{k} - y_{k-1}\vert{}}{\vert{}y_{k}\vert{}} < \epsilon$$


*(Onde $y_k = \left(1 + \frac{x}{n_k}\right)^{n_k}$ e $\epsilon = 10^{-7}$, por exemplo).*
* **Estratégia Complementar (Inversão de Sinal):**
Para $x < 0$, elimina-se a subtração na base calculando o inverso do valor positivo:
$$e^x = \frac{1}{e^{\vert{}x\vert{}}} \approx \frac{1}{\left(1 + \frac{\vert{}x\vert{}}{n}\right)^n}$$


Dessa forma, evita-se a subtração de termos próximos e o critério de erro relativo pode ser aplicado com total estabilidade numérica.

## Atividade 3

Para $|x|$ grande, use:
$$
e^x = \left(e^{m\cdot 2^{-k}}\right)^{2^k}, \quad
k = \left\lceil \log_2\!\left(\frac{|x|}{\theta}\right)\right\rceil, \quad m = \frac{x}{2^k}
$$
Calcule $e^{m}$ pela série (Ex. 1) e depois eleve ao quadrado $k$ vezes.

In [3]:
import math


def exp_serie_taylor(m: float, tol: float = 1e-12, max_iter: int = 100) -> float:
    soma = 1.0
    termo = 1.0
    for i in range(1, max_iter):
        termo *= m / i
        soma += termo
        if abs(termo) < tol:
            break
    return soma


def exp_reducao_argumento(x: float, theta: float = 0.5) -> float:
    if abs(x) < theta:
        return exp_serie_taylor(x)

    k = math.ceil(math.log2(abs(x) / theta))
    if k < 0:
        k = 0

    m = x / (2**k)

    resultado = exp_serie_taylor(m)
    for _ in range(k):
        resultado = resultado**2

    return resultado


def main():
    x = 10.0
    print(exp_reducao_argumento(x))


if __name__ == "__main__":
    main()

22026.46579480578


## Atividade 4

Use:
$$
\cos x=\sum_{n=0}^{\infty}(-1)^n\frac{x^{2n}}{(2n)!}
$$
com a recursão:
$$
t_{n+1}=t_n\cdot\frac{-x^2}{(2n+1)(2n+2)}
$$
Defina um critério de parada baseado em `epsilon` e compare o erro relativo para $x\in[-20,20]$ (200 pontos) contra `math.cos(x)`.

In [8]:
import math


def cos_serie(x: float, eps: float = 1e-12, max_iter: int = 200) -> float:
    x = math.fmod(x, 2 * math.pi)
    if x > math.pi:
        x -= 2 * math.pi
    elif x < -math.pi:
        x += 2 * math.pi

    termo = 1.0
    soma = 1.0

    for n in range(max_iter):
        termo *= - (x ** 2) / ((2 * n + 1) * (2 * n + 2))
        soma += termo

        if abs(termo) < eps:
            break

    return soma


def main():
    inicio, fim, num_pontos = -20.0, 20.0, 200
    passo = (fim - inicio) / (num_pontos - 1)
    pontos_x = [inicio + i * passo for i in range(num_pontos)]
    eps = 1e-12

    print(f"{'x':>10} | {'Série Cos':>18} | {'math.cos':>18} | {'Erro Relativo':>18}")
    print("-" * 72)

    for x in pontos_x:
        val_serie = cos_serie(x, eps=eps)
        val_exact = math.cos(x)

        if abs(val_exact) > 1e-12:
            erro_rel = abs(val_serie - val_exact) / abs(val_exact)
        else:
            erro_rel = abs(val_serie - val_exact)

        print(f"{x:10.4f} | {val_serie:18.12f} | {val_exact:18.12f} | {erro_rel:18.4e}")


if __name__ == "__main__":
    main()

         x |          Série Cos |           math.cos |      Erro Relativo
------------------------------------------------------------------------
  -20.0000 |     0.408082061813 |     0.408082061813 |         3.2647e-15
  -19.7990 |     0.582139280638 |     0.582139280638 |         7.6286e-16
  -19.5980 |     0.732755398358 |     0.732755398358 |         1.5151e-15
  -19.3970 |     0.853865530230 |     0.853865530230 |         3.9007e-16
  -19.1960 |     0.940592914040 |     0.940592914040 |         2.3607e-16
  -18.9950 |     0.989445283522 |     0.989445283522 |         1.1221e-16
  -18.7940 |     0.998455492032 |     0.998455492032 |         0.0000e+00
  -18.5930 |     0.967260723976 |     0.967260723976 |         1.1478e-16
  -18.3920 |     0.897117104369 |     0.897117104369 |         6.1877e-16
  -18.1910 |     0.790849118231 |     0.790849118231 |         5.6153e-16
  -17.9899 |     0.652735876578 |     0.652735876578 |         1.0205e-15
  -17.7889 |     0.488338808725 |     0

## Atividade 5

Dado $x$ e uma tolerância $\tau$, encontre o menor $N$ tal que:
$$
R_{N+1}(x)=\sum_{n=N+1}^{\infty}\frac{|x|^n}{n!} < \tau
$$

In [9]:
import math


def encontrar_menor_N(x: float, tau: float = 1e-6, max_iter: int = 1000) -> int:
    abs_x = abs(x)
    termo = (abs_x ** 1) / math.factorial(1)
    
    # R_1(x) = sum_{n=1}^{inf} |x|^n / n! = e^{|x|} - 1
    resto_atual = math.exp(abs_x) - 1.0
    
    N = 0
    while resto_atual >= tau and N < max_iter:
        resto_atual -= termo
        N += 1
        termo *= abs_x / (N + 1)
        
    return N


def main():
    x = 2.0
    tau = 1e-6
    N_min = encontrar_menor_N(x, tau)
    
    print(f"Para x = {x} e tau = {tau}:")
    print(f"Menor N = {N_min}")


if __name__ == "__main__":
    main()

Para x = 2.0 e tau = 1e-06:
Menor N = 13


## Atividade 6

Usando `decimal` ou `mpmath`, compute $e^x$ em alta precisão e compare com o resultado de `float64` (Ex. 1) para $x\in\{20, 40, 50\}$.
Analise:
- perda de dígitos significativos;
- quando o `float64` começa a saturar por overflow.


In [10]:
from decimal import Decimal, getcontext
import math

getcontext().prec = 50


def exp_float64(x: float, max_iter: int = 200) -> float:
    soma = 1.0
    termo = 1.0
    for n in range(1, max_iter):
        termo *= x / n
        soma += termo
        if abs(termo) < 1e-16 * abs(soma):
            break
    return soma


def exp_decimal(x: int) -> Decimal:
    x_dec = Decimal(x)
    soma = Decimal(1)
    termo = Decimal(1)
    for n in range(1, 200):
        termo *= x_dec / Decimal(n)
        soma += termo
        if abs(termo) < Decimal("1e-45"):
            break
    return soma


def main():
    valores_x = [20, 40, 50]

    print(
        f"{'x':>5} | {'float64 (Ex. 1)':>22} | {'Alta Precisão (Decimal)':>30} | {'Erro Relativo':>15}"
    )
    print("-" * 80)

    for x in valores_x:
        val_f64 = exp_float64(float(x))
        val_dec = exp_decimal(x)

        val_f64_dec = Decimal(str(val_f64))
        erro_relativo = abs(val_f64_dec - val_dec) / val_dec

        print(
            f"{x:5d} | {val_f64:22.10e} | {val_dec:30.15e} | {float(erro_relativo):15.4e}"
        )


if __name__ == "__main__":
    main()

    x |        float64 (Ex. 1) |        Alta Precisão (Decimal) |   Erro Relativo
--------------------------------------------------------------------------------
   20 |       4.8516519541e+08 |           4.851651954097903e+8 |      3.7519e-16
   40 |       2.3538526684e+17 |          2.353852668370200e+17 |      3.1689e-16
   50 |       5.1847055286e+21 |          5.184705528587072e+21 |      1.6464e-15


### 1. Perda de Dígitos Significativos

O tipo de dado `float64` (padrão IEEE 754 de precisão dupla) reserva 52 bits para a mantissa, o que proporciona um limite fixo de **15 a 17 dígitos decimais de precisão**:

* **Acúmulo de Arredondamento:** Ao calcular $e^x$ para valores crescentes ($x = 20, 40, 50$), os termos intermediários da série de Taylor ($\frac{x^n}{n!}$) tornam-se consideravelmente grandes antes de começarem a decair. A soma sucessiva desses termos grandes resulta na perda gradual dos últimos bits de precisão por arredondamento.
* **Comparação de Precisão:** Enquanto o `float64` preserva exatidão apenas até a 15ª casa significativa, bibliotecas de alta precisão como `decimal` ou `mpmath` mantêm dezenas de dígitos exatos (configurados no contexto), revelando os pequenos desvios acumulados pela aritmética de ponto flutuante padrão.

---

### 2. Saturação por Overflow no `float64`

O valor máximo representável em ponto flutuante de 64 bits é limitado pela sua representação binária:

* **Limite do `float64`:** O maior valor finito permissível é aproximadamente $1,79769 \times 10^{308}$.
* **Ponto Crítico de Saturação:** O estouro de memória (*overflow*) ocorre quando a função exponencial excede este limite superior:

$$x_{\text{limite}} = \ln(1,79769 \times 10^{308}) \approx 709,78$$

* **Conclusão:** Para valores de **$x \ge 710$**, o cálculo em `float64` satura completamente, resultando no valor simbólico `inf` (*Infinity*), enquanto `decimal` e `mpmath` continuam computando o resultado corretamente graças ao gerenciamento dinâmico de memória.

## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 3"
git push origin semana3
```